In [0]:
from pyspark.sql.functions import current_timestamp, date_trunc, md5
from pyspark.sql import functions as F
from datetime import datetime
from delta.tables import DeltaTable

# Widget with default value
dbutils.widgets.text("table_name", "dim_states")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
gold_base = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Order: code, name, region, lat, lng
state_data = [
    ('AC', 'Acre', 'North', -9.0238, -70.8120),
    ('AL', 'Alagoas', 'Northeast', -9.5713, -36.7820),
    ('AM', 'Amazonas', 'North', -3.4168, -65.8561),
    ('AP', 'Amapá', 'North', 1.4154, -51.7711),
    ('BA', 'Bahia', 'Northeast', -12.5127, -41.7007),
    ('CE', 'Ceará', 'Northeast', -5.2000, -39.5000),
    ('DF', 'Distrito Federal', 'Midwest', -15.7998, -47.8645),
    ('ES', 'Espírito Santo', 'Southeast', -19.1834, -40.3089),
    ('GO', 'Goiás', 'Midwest', -15.8270, -49.8362),
    ('MA', 'Maranhão', 'Northeast', -5.4200, -45.4400),
    ('MG', 'Minas Gerais', 'Southeast', -18.5122, -44.5550),
    ('MS', 'Mato Grosso do Sul', 'Midwest', -20.7722, -54.7852),
    ('MT', 'Mato Grosso', 'Midwest', -12.6819, -56.9211),
    ('PA', 'Pará', 'North', -3.7922, -52.4818),
    ('PB', 'Paraíba', 'Northeast', -7.2400, -36.7800),
    ('PE', 'Pernambuco', 'Northeast', -8.2833, -37.9833),
    ('PI', 'Piauí', 'Northeast', -7.7183, -42.7289),
    ('PR', 'Paraná', 'South', -24.8923, -51.5597),
    ('RJ', 'Rio de Janeiro', 'Southeast', -22.4474, -42.9912),
    ('RN', 'Rio Grande do Norte', 'Northeast', -5.4026, -36.9541),
    ('RO', 'Rondônia', 'North', -10.8300, -63.3400),
    ('RR', 'Roraima', 'North', 2.1351, -61.3231),
    ('RS', 'Rio Grande do Sul', 'South', -30.0019, -53.7611),
    ('SC', 'Santa Catarina', 'South', -27.2423, -50.2189),
    ('SE', 'Sergipe', 'Northeast', -10.5741, -37.3857),
    ('SP', 'São Paulo', 'Southeast', -23.5505, -46.6333),
    ('TO', 'Tocantins', 'North', -10.1753, -48.2982)
]

# Schema Definition
schema = "state_code STRING, state_name STRING, region STRING, center_lat DOUBLE, center_lng DOUBLE"

# Creating DataFrame and Adding Metadata
df_states = (spark.createDataFrame(state_data, schema)
             .withColumn("state_key", md5(F.col("state_code")))
             .withColumn("gold_load_at", date_trunc("second", current_timestamp())))

# Save to Gold
gold_path = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/dim_states"

if DeltaTable.isDeltaTable(spark, gold_path):
    print("--> [Merge] Upserting records to dim_states to prevent duplicates.")
    dt = DeltaTable.forPath(spark, gold_path)

    (dt.alias("target")
     .merge(df_states.alias("source"), "target.state_key = source.state_key")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    print("--> [INIT] Creating dim_states table for the first time.")
    df_states.write.format("delta").save(gold_path)

message = f"--> Loaded dim_states successfully with records: {df_states.count()}"
print(message)
dbutils.notebook.exit(message)